In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import csv
import os

if not os.path.exists("test_notebooks"):
    os.chdir("..")

assert os.path.exists("test_notebooks")

In [46]:
# benchmark dataset
# 目标: 12个tag进行rag，3秒内
import random

from entropy.domain.services.tag_checker import TagChecker


input_data = """
shimmering, pleading, soaked, slightly open mouth, thin straps, sheer dress, blushing, shoulderless dress,
seifuku, japanese school uniform, bubbles, knitwear,
low angle, daydreaming, wild life, surrounded by bubbles, sunbeams, cross-legged, flowing dress,
adjusting glove, rim light, tyndall effect, floating flower petals, misty,
magical atmosphere, motes, thoughtful, dangling legs, cinematic lighting, purple and gold theme,
reading book, white clouds, large bubbles, scenic
"""

input_data = TagChecker.extract_all_tags(input_data)
input_data = list(set(input_data))
input_data = sorted(input_data)

input_data = random.Random(6).sample(input_data, k=12)

assert len(input_data) == 12

print(",".join(input_data))

dangling legs,tyndall effect,pleading,blushing,adjusting glove,cross-legged,seifuku,reading book,motes,shoulderless dress,shimmering,large bubbles


In [47]:
from entropy.domain.services.rag_service import RagService

In [48]:
# 非batch

# result1 = []

# for invalid_tag in input_data:
#     tags, scores = RagService.do_rag(query_text=invalid_tag, recall_count=20, rerank_output=10)
#     result1.append(tags)

batch_rag_output = RagService.batch_rag(query_text_list=input_data,recall_count=20, rerank_output=10)

result1 = [tags for tags, scores in batch_rag_output]

Compute Scores: 100%|██████████| 2/2 [00:00<00:00,  4.69it/s]


In [57]:
# batch
batch_rag_output = RagService.batch_rag_simple(query_text_list=input_data,recall_count=10)

print(batch_rag_output[0])

result2 = [tags for tags, scores in batch_rag_output]

(['dangling', 'holding legs', 'hanging legs', 'swinging legs', 'outstretched legs', 'separated legs', 'wiggling toes', 'legs apart', 'legs together', 'dangling eye'], [0.11924499273300171, 0.1539076566696167, 0.1542297601699829, 0.1572718620300293, 0.17824316024780273, 0.17972064018249512, 0.1816970705986023, 0.1831880807876587, 0.18372613191604614, 0.18536406755447388])


In [ ]:
time_compare = False

if time_compare:
    for invalid_tag in input_data:
        RagService.rag_simple(query_text=invalid_tag, recall_count=10)

In [55]:
import json



for (invalid_tag, r1, r2) in zip(input_data, result1, result2):
    if not json.dumps(r1) == json.dumps(r2):
        print(f"invalid danbooru tag: {invalid_tag}")
        print("rag result 1:", r1)
        print("rag result 2:", r2)

invalid danbooru tag: dangling legs
rag result 1: ['foot dangle', 'dangling', 'hanging legs', 'holding legs', 'wiggling', 'holding leg', 'swinging legs', 'legs together', 'detached legs', 'outstretched legs']
rag result 2: ['dangling', 'holding legs', 'hanging legs', 'swinging legs', 'outstretched legs', 'separated legs', 'wiggling toes', 'legs apart', 'legs together', 'dangling eye']
invalid danbooru tag: tyndall effect
rag result 1: ['tidal wave', 'tryndamere', 'tynamo', 'tingle', 'thymilph', 'tianel ent', 'tyrantrum', 'mindoll', 'dulldull', 'thranduil']
rag result 2: ['mindoll', 'tidal wave', 'tianel ent', 'dulldull', 'tianzi', 'thimble', 'tatl', 'tyrantrum', 'thranduil', 'tynamo']
invalid danbooru tag: pleading
rag result 1: ['praying', 'begging', 'arguing', 'apologizing', 'howling', 'inasaba', 'talking', 'a lone prayer', 'screaming', 'mourning']
rag result 2: ['praying', 'begging', 'pleading face emoji', 'arguing', 'moaning', 'apologizing', 'inasaba', 'kneeling', 'shouting', 'cryi

In [ ]:
"""
我试图从方法1改为方法2，请帮我看看效果是否有明显下降。从2个角度：

1. 这个是检测到无效tag，对用户输出guess you like tags，帮助用户找到原本想要找的哪个

2. 除了将无效变为有效之外，tag也能提供灵感

"""
print()

## 实验结论

### rerank 一定好吗
recall -> 500 -> rerank -> 10

recall -> 10

前者效果差，后者好（经过gemini和豆包反复看case）

也可能是因为recall没有卡阈值造成的，但我懒得卡，不想尝试了

### 尝试优化方法1

recall -> 20 -> rerank -> 10 // 500改成20

recall -> 10

实验结果：还是方法2更好。

### 最终结论
所以最终结论是：这个耗时的reranker费力不讨好。

既然这样，bm25也没必要支持了，因为bm25肯定比embedding差；bm25+reranker也一定比embedding差。用户花30分钟embedding建库没啥大问题。

In [ ]:
#